# Time Series Analysis of Economic Development Dynamics in Central America  
### Second Notebook: Data Cleaning, Preprocessing, and Time Series Preparation  

This second notebook builds directly on the exploratory work conducted in Notebook 1 and focuses on preparing the dataset for formal time series analysis. While the previous step emphasized understanding the structure, distributions, and relationships within the data, this stage aims to ensure that the dataset is fully consistent, clean, and properly structured for dynamic analysis over time.

Given the longitudinal nature of the dataset, particular attention is given to data cleaning, handling of missing values, and ensuring temporal consistency across countries and indicators. Additional preprocessing steps are also introduced to prepare the data for time series methods, including the construction of standardized formats and the alignment of observations across years.

The objective of this notebook is not yet to perform advanced econometric or forecasting analysis, but rather to establish a reliable analytical foundation. This includes ensuring that trends are not distorted by data quality issues and that all variables are appropriately formatted for subsequent time-based modeling and interpretation.

By the end of this stage, the dataset will be fully structured and ready for time series analysis, enabling robust examination of growth dynamics, structural changes, and cross-country comparisons in the following notebook.

**Author**: J-F Jutras  
**Date**: April 2026  
**Dataset**: UN National Accounts Main Aggregates Database (Central America subset)

## 2.1-Data Loading and Overview

In [1]:
import pandas as pd
import sys

# File path (Kaggle dataset)
file_path = "/kaggle/input/datasets/jfjutras07/ca-timeseries/CA_Timeseries.csv"

# Load dataset
df = pd.read_csv(file_path)

# Load custom library from GitHub
!rm -rf /kaggle/working/jfj-utils
!git clone https://github.com/jfjutras07/jfj-utils.git

# Add to Python path
sys.path.append("/kaggle/working/jfj-utils")

# Preview
display(df.head())
display(df.tail())
df.info()

Cloning into 'jfj-utils'...
remote: Enumerating objects: 3527, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 3527 (delta 211), reused 87 (delta 87), pack-reused 3260 (from 4)
Receiving objects: 100% (3527/3527), 1.18 MiB | 6.76 MiB/s, done.
Resolving deltas: 100% (2322/2322), done.


,Country,Year,Population,Gross Domestic Product (GDP),Per capita GNI,Gross capital formation,Exports of goods and services,Imports of goods and services,Final consumption expenditure
0,Belize,1970,120905,27917782.0,195,7450178.0,10317179.0,12335045.0,21072623.0
1,Belize,1971,123091,35385285.0,245,9299593.0,13474524.0,15709957.0,26315597.0
2,Belize,1972,125054,46000141.0,315,12422347.0,18383266.0,21233992.0,33738204.0
3,Belize,1973,126875,74809672.0,514,20028373.0,24555373.0,31414706.0,58898086.0
4,Belize,1974,128620,99801545.0,639,25015682.0,41368811.0,44947663.0,70891032.0


,Country,Year,Population,Gross Domestic Product (GDP),Per capita GNI,Gross capital formation,Exports of goods and services,Imports of goods and services,Final consumption expenditure
359,Panama,2017,4096063,6.220273e+10,14074,2.595785e+10,2.600303e+10,2.837949e+10,3.862134e+10
360,Panama,2018,4165255,6.492941e+10,14320,2.693338e+10,2.778392e+10,3.064638e+10,4.085849e+10
361,Panama,2019,4232532,6.698443e+10,14596,2.565666e+10,2.760294e+10,2.932200e+10,4.304682e+10
362,Panama,2020,4294396,5.397704e+10,12048,1.298641e+10,2.137917e+10,1.860550e+10,3.821697e+10
363,Panama,2021,4351267,6.360507e+10,14012,1.622814e+10,3.247930e+10,2.648142e+10,4.137905e+10


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364 entries, 0 to 363
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        364 non-null    object 
 1   Year                           364 non-null    int64  
 2   Population                     364 non-null    int64  
 3   Gross Domestic Product (GDP)   364 non-null    float64
 4   Per capita GNI                 364 non-null    int64  
 5   Gross capital formation        364 non-null    float64
 6   Exports of goods and services  364 non-null    float64
 7   Imports of goods and services  364 non-null    float64
 8   Final consumption expenditure  364 non-null    float64
dtypes: float64(5), int64(3), object(1)
memory usage: 25.7+ KB


## 2.2-Data Cleaning and Transforming

### Time Series Structuring

Time series analysis in a panel data context requires a well-defined and consistent temporal index to preserve chronological ordering and ensure the validity of lagged, differenced, and rolling computations. 

Because the dataset is annual, using a specialized temporal representation such as a yearly PeriodIndex is more appropriate than a full datetime format, as it avoids introducing artificial intra-year structure (e.g., arbitrary day and month values).

Structuring the dataset as a country–year panel with a hierarchical index ensures that each country's time series is processed independently while maintaining proper temporal alignment across all countries. This is essential for both econometric modeling and machine learning applications involving panel time series data.

In [2]:
# Convert Year to PeriodIndex (annual frequency)
df["Year"] = pd.PeriodIndex(df["Year"], freq="Y")

# Set multi-index (Country, Year)
df = df.set_index(["Country", "Year"])

# Sort index (critical for time series operations)
df = df.sort_index()

# Validate index uniqueness (critical for panel data)
assert df.index.is_unique, "Duplicate Country-Year pairs detected"

# Quick check
print(df.index.names)
display(df.head())

['Country', 'Year']


Population  Gross Domestic Product (GDP)  Per capita GNI  \
Country Year                                                             
Belize  1970      120905                    27917782.0             195   
        1971      123091                    35385285.0             245   
        1972      125054                    46000141.0             315   
        1973      126875                    74809672.0             514   
        1974      128620                    99801545.0             639   

              Gross capital formation  Exports of goods and services  \
Country Year                                                           
Belize  1970                7450178.0                     10317179.0   
        1971                9299593.0                     13474524.0   
        1972               12422347.0                     18383266.0   
        1973               20028373.0                     24555373.0   
        1974               25015682.0                     41368811.0   

              Imports of goods and services  Final consumption expenditure  
Country Year                                                                
Belize  1970                     12335045.0                     21072623.0  
        1971                     15709957.0                     26315597.0  
        1972                     21233992.0                     33738204.0  
        1973                     31414706.0                     58898086.0  
        1974                     44947663.0                     70891032.0

### Checking Temporal Completeness

Time series analysis requires consistent temporal spacing to ensure the validity of growth rates, lagged variables, and rolling statistics. In a panel data context, this also guarantees that all countries remain comparable over the same time horizon.

To verify temporal completeness, we define the full expected range of years and compare it to the observed years for each country. Identifying missing periods is critical, as gaps in the time index can distort dynamic analysis and lead to biased results if left unaddressed.

In [3]:
# Define expected full range of years (annual PeriodIndex)
expected_years = pd.period_range(start="1970", end="2021", freq="Y")

# Check missing years per country
missing_summary = {}

for country in df.index.get_level_values(0).unique():
    actual_years = df.loc[country].index
    missing_years = expected_years.difference(actual_years)
    
    missing_summary[country] = len(missing_years)
    print(f"{country}: {len(missing_years)} missing years")

# Detailed check (print missing years for countries with gaps)
for country, count in missing_summary.items():
    if count > 0:
        actual_years = df.loc[country].index
        missing_years = expected_years.difference(actual_years)
        print(f"\n{country} missing years:", list(missing_years))

Belize: 0 missing years
Costa Rica: 0 missing years
El Salvador: 0 missing years
Guatemala: 0 missing years
Honduras: 0 missing years
Nicaragua: 0 missing years
Panama: 0 missing years


### Panel Integrity Validation

In [4]:
# The dataset has already been structured as a Country-Year panel
# using a MultiIndex and verified for completeness.
# Here we only re-confirm structural validity before feature engineering.

assert df.index.is_unique, "Duplicate Country-Year observations detected"

print("Panel structure confirmed: Country-Year index is unique and properly set.")

Panel structure confirmed: Country-Year index is unique and properly set.


In [5]:
# We verify that all macroeconomic variables are correctly typed.
# This ensures that numerical transformations (log, growth rates, lags)
# will behave correctly in subsequent steps.

print(df.dtypes)

Population                         int64
Gross Domestic Product (GDP)     float64
Per capita GNI                     int64
Gross capital formation          float64
Exports of goods and services    float64
Imports of goods and services    float64
Final consumption expenditure    float64
dtype: object


All macroeconomic indicators are correctly stored as numerical types (int or float), which ensures compatibility with time series transformations such as logarithmic scaling, differencing, and lag feature construction. No categorical or malformed numeric fields are present, confirming that the dataset is technically ready for quantitative time series operations.

In [6]:
# We compute the total number of missing values per variable.
# This helps identify whether missingness is concentrated in specific indicators,
# which could bias economic interpretation or dynamic modeling.

missing_summary = df.isna().sum().sort_values(ascending=False)

print("Missing values per variable:")
print(missing_summary)

Missing values per variable:
Population                       0
Gross Domestic Product (GDP)     0
Per capita GNI                   0
Gross capital formation          0
Exports of goods and services    0
Imports of goods and services    0
Final consumption expenditure    0
dtype: int64


The dataset has been successfully structured into a clean country-year panel time series with a validated MultiIndex, ensuring correct temporal ordering and uniqueness of observations. Temporal completeness has been verified across all Central American countries for the full 1970–2021 period, confirming a balanced panel without gaps. All variables are consistently formatted as numeric types, and a full diagnostic of missing values indicates a complete dataset with no imputation required. Overall, the data is now structurally sound and fully prepared for the next stage of time series feature engineering.

### Apply Log Transformation

In [7]:
# Before applying log transformations, we must ensure that all variables are strictly positive.
# Logarithmic functions are undefined for zero or negative values.
# This step prevents numerical errors and ensures mathematical validity.

log_vars = [
    "Gross Domestic Product (GDP)",
    "Gross capital formation",
    "Exports of goods and services",
    "Imports of goods and services",
    "Final consumption expenditure",
    "Per capita GNI"
]

# Check minimum values per variable
min_values = df[log_vars].min()

print("Minimum values per variable:")
print(min_values)

# Identify problematic variables
problematic_vars = min_values[min_values <= 0]

print("\nVariables with non-positive values:")
print(problematic_vars)

Minimum values per variable:
Gross Domestic Product (GDP)      27917782.0
Gross capital formation         -105872604.0
Exports of goods and services     10317179.0
Imports of goods and services     12335045.0
Final consumption expenditure     21072623.0
Per capita GNI                         195.0
dtype: float64

Variables with non-positive values:
Gross capital formation   -105872604.0
dtype: float64


In [8]:
# Gross capital formation contains negative values, which makes log transformation impossible.
# In macroeconomic accounting, negative values may reflect net disinvestment or statistical adjustments.
# Therefore, we do not modify the data at this stage.

# Strategy:
# - Keep Gross capital formation in levels for growth rate analysis
# - Exclude it from log-transformed feature space
# - Ensure consistency in later dynamic modeling steps

problem_variable = "Gross capital formation"
print(f"Excluded from log transformation: {problem_variable}")

Excluded from log transformation: Gross capital formation


In [9]:
import numpy as np

# Log transformation is applied only to variables that are strictly positive.
# This ensures mathematical validity and avoids introducing infinite or undefined values.

log_vars = [
    "Gross Domestic Product (GDP)",
    "Exports of goods and services",
    "Imports of goods and services",
    "Final consumption expenditure",
    "Per capita GNI"
]

for col in log_vars:
    df[f"log_{col}"] = np.log(df[col])

### Log Transformation Validation

In [10]:
# We ensure that log-transformed variables did not introduce NaN or infinite values.
# This step is critical to confirm numerical stability after transformation.

log_cols = [col for col in df.columns if col.startswith("log_")]

invalid_counts = df[log_cols].isna().sum() + np.isinf(df[log_cols]).sum()

print("Invalid values in log-transformed variables:")
print(invalid_counts)

Invalid values in log-transformed variables:
log_Gross Domestic Product (GDP)     0
log_Exports of goods and services    0
log_Imports of goods and services    0
log_Final consumption expenditure    0
log_Per capita GNI                   0
dtype: int64


In [11]:
# We compute skewness to verify whether log transformation
# successfully reduced right-skewness typical of macroeconomic variables.

skewness_before = df[[
    "Gross Domestic Product (GDP)",
    "Exports of goods and services",
    "Imports of goods and services",
    "Final consumption expenditure",
    "Per capita GNI"
]].skew()

skewness_after = df[log_cols].skew()

print("Skewness BEFORE log transformation:")
print(skewness_before)

print("\nSkewness AFTER log transformation:")
print(skewness_after)

Skewness BEFORE log transformation:
Gross Domestic Product (GDP)     2.155147
Exports of goods and services    2.229384
Imports of goods and services    1.776744
Final consumption expenditure    2.287244
Per capita GNI                   2.293782
dtype: float64

Skewness AFTER log transformation:
log_Gross Domestic Product (GDP)    -0.690824
log_Exports of goods and services   -0.567012
log_Imports of goods and services   -0.606341
log_Final consumption expenditure   -0.749332
log_Per capita GNI                   0.184755
dtype: float64


### Growth Rate Construction

In [12]:
# Growth rates are computed within each country to preserve the panel structure.
# This ensures that each time series evolves independently over time,
# avoiding any cross-country contamination.

growth_vars = [
    "Gross Domestic Product (GDP)",
    "Gross capital formation",
    "Exports of goods and services",
    "Imports of goods and services",
    "Final consumption expenditure",
    "Per capita GNI"
]

for col in growth_vars:
    df[f"growth_{col}"] = df.groupby(level=0)[col].pct_change()

In [13]:
# We inspect the newly created growth variables to ensure correct generation
# and to verify expected presence of NaN values in the first time period per country.

growth_cols = [c for c in df.columns if c.startswith("growth_")]

df[growth_cols].head()

growth_Gross Domestic Product (GDP)  \
Country Year                                        
Belize  1970                                  NaN   
        1971                             0.267482   
        1972                             0.299979   
        1973                             0.626292   
        1974                             0.334073   

              growth_Gross capital formation  \
Country Year                                   
Belize  1970                             NaN   
        1971                        0.248238   
        1972                        0.335795   
        1973                        0.612286   
        1974                        0.249012   

              growth_Exports of goods and services  \
Country Year                                         
Belize  1970                                   NaN   
        1971                              0.306028   
        1972                              0.364298   
        1973                              0.335746   
        1974                              0.684715   

              growth_Imports of goods and services  \
Country Year                                         
Belize  1970                                   NaN   
        1971                              0.273604   
        1972                              0.351626   
        1973                              0.479454   
        1974                              0.430784   

              growth_Final consumption expenditure  growth_Per capita GNI  
Country Year                                                               
Belize  1970                                   NaN                    NaN  
        1971                              0.248805               0.256410  
        1972                              0.282061               0.285714  
        1973                              0.745739               0.631746  
        1974                              0.203622               0.243191

### Create Lag Features

In [14]:
# We construct 1-year lag features within each country.
# This preserves the panel structure and ensures that time dependence
# is modeled correctly within each economic system.

lag_vars = [
    "Gross Domestic Product (GDP)",
    "Gross capital formation",
    "Exports of goods and services",
    "Imports of goods and services",
    "Final consumption expenditure",
    "Per capita GNI"
]

for col in lag_vars:
    df[f"lag1_{col}"] = df.groupby(level=0)[col].shift(1)

In [15]:
# We also create lagged growth rates to capture momentum effects
# in economic dynamics (e.g., persistence of expansion or recession phases).

growth_cols = [c for c in df.columns if c.startswith("growth_")]

for col in growth_cols:
    df[f"lag1_{col}"] = df.groupby(level=0)[col].shift(1)

In [16]:
# We inspect lagged variables to ensure correct alignment
# and expected NaN patterns at the beginning of each country series.

lag_cols = [c for c in df.columns if c.startswith("lag1_")]

df[lag_cols].head()

lag1_Gross Domestic Product (GDP)  lag1_Gross capital formation  \
Country Year                                                                    
Belize  1970                                NaN                           NaN   
        1971                         27917782.0                     7450178.0   
        1972                         35385285.0                     9299593.0   
        1973                         46000141.0                    12422347.0   
        1974                         74809672.0                    20028373.0   

              lag1_Exports of goods and services  \
Country Year                                       
Belize  1970                                 NaN   
        1971                          10317179.0   
        1972                          13474524.0   
        1973                          18383266.0   
        1974                          24555373.0   

              lag1_Imports of goods and services  \
Country Year                                       
Belize  1970                                 NaN   
        1971                          12335045.0   
        1972                          15709957.0   
        1973                          21233992.0   
        1974                          31414706.0   

              lag1_Final consumption expenditure  lag1_Per capita GNI  \
Country Year                                                            
Belize  1970                                 NaN                  NaN   
        1971                          21072623.0                195.0   
        1972                          26315597.0                245.0   
        1973                          33738204.0                315.0   
        1974                          58898086.0                514.0   

              lag1_growth_Gross Domestic Product (GDP)  \
Country Year                                             
Belize  1970                                       NaN   
        1971                                       NaN   
        1972                                  0.267482   
        1973                                  0.299979   
        1974                                  0.626292   

              lag1_growth_Gross capital formation  \
Country Year                                        
Belize  1970                                  NaN   
        1971                                  NaN   
        1972                             0.248238   
        1973                             0.335795   
        1974                             0.612286   

              lag1_growth_Exports of goods and services  \
Country Year                                              
Belize  1970                                        NaN   
        1971                                        NaN   
        1972                                   0.306028   
        1973                                   0.364298   
        1974                                   0.335746   

              lag1_growth_Imports of goods and services  \
Country Year                                              
Belize  1970                                        NaN   
        1971                                        NaN   
        1972                                   0.273604   
        1973                                   0.351626   
        1974                                   0.479454   

              lag1_growth_Final consumption expenditure  \
Country Year                                              
Belize  1970                                        NaN   
        1971                                        NaN   
        1972                                   0.248805   
        1973                                   0.282061   
        1974                                   0.745739   

              lag1_growth_Per capita GNI  
Country Year                              
Belize  1970                         NaN  
        1971                         NaN  
        197

The feature engineering phase transformed the dataset into a fully dynamic panel time series structure. We constructed log-transformed variables to stabilize variance and reduce skewness, while carefully excluding non-positive series from log space. Year-over-year growth rates were then computed to capture economic dynamics across countries, followed by the creation of lagged features to model temporal dependence and economic inertia. Together, these transformations establish a multi-layered dataset ready for econometric and time series analysis.

## 2.3-Temporal Cleaning and Validating Model Structure

In [17]:
# We inspect missing values introduced by time-dependent transformations
# (lags and growth rates) to understand their structure.
# These NaNs are expected at the beginning of each country’s time series.

missing_summary = df.isna().sum().sort_values(ascending=False)

print("Missing values per variable:")
print(missing_summary)

Missing values per variable:
lag1_growth_Gross capital formation          14
lag1_growth_Gross Domestic Product (GDP)     14
lag1_growth_Final consumption expenditure    14
lag1_growth_Per capita GNI                   14
lag1_growth_Imports of goods and services    14
lag1_growth_Exports of goods and services    14
growth_Imports of goods and services          7
growth_Exports of goods and services          7
growth_Gross capital formation                7
growth_Gross Domestic Product (GDP)           7
lag1_Gross Domestic Product (GDP)             7
lag1_Gross capital formation                  7
lag1_Exports of goods and services            7
lag1_Imports of goods and services            7
growth_Per capita GNI                         7
growth_Final consumption expenditure          7
lag1_Final consumption expenditure            7
lag1_Per capita GNI                           7
log_Final consumption expenditure             0
log_Per capita GNI                            0
log_Exports

The missing values observed are entirely structural and arise from the sequential construction of growth rates and lagged features within a country-level panel time series, confirming that the temporal transformations were applied correctly without introducing data leakage.

In [18]:
# We organize variables into conceptual groups to better control redundancy
# and ensure interpretability in later econometric or ML models.

level_vars = [
    "Gross Domestic Product (GDP)",
    "Gross capital formation",
    "Exports of goods and services",
    "Imports of goods and services",
    "Final consumption expenditure",
    "Per capita GNI",
    "Population"
]

log_vars = [f"log_{col}" for col in level_vars if f"log_{col}" in df.columns]

growth_vars = [c for c in df.columns if c.startswith("growth_") and "lag1" not in c]

lag_vars = [c for c in df.columns if c.startswith("lag1_") and "growth" not in c]

### Create Analysis-Ready Dataset

In [19]:
# We define a minimal, interpretable feature set combining:
# - logs (for scale normalization)
# - growth rates (for dynamics)
# - lagged GDP (as key state variable)

df_core = df[
    log_vars +
    growth_vars +
    ["lag1_Gross Domestic Product (GDP)"]
].copy()

In [20]:
print(df_core.shape)
print(df_core.columns)

(364, 12)
Index(['log_Gross Domestic Product (GDP)', 'log_Exports of goods and services',
       'log_Imports of goods and services',
       'log_Final consumption expenditure', 'log_Per capita GNI',
       'growth_Gross Domestic Product (GDP)', 'growth_Gross capital formation',
       'growth_Exports of goods and services',
       'growth_Imports of goods and services',
       'growth_Final consumption expenditure', 'growth_Per capita GNI',
       'lag1_Gross Domestic Product (GDP)'],
      dtype='object')


The consolidated feature set is structurally sound and well-balanced, combining log-scaled levels, growth dynamics, and a key lagged state variable without redundancy, making it suitable for robust econometric time series analysis.

In [21]:
# We remove observations with incomplete temporal information
# (introduced by lag and growth construction).
# This ensures a fully aligned modeling dataset.

df_model_final = df_core.dropna()

In [22]:
print(df_model_final.shape)
print(df_model_final.isna().sum().sum())

(357, 12)
0


In [23]:
# The dataset is now fully:
# - time-consistent
# - panel-structured
# - feature-engineered
# - free of missing dynamic values

df_model_final.head()

log_Gross Domestic Product (GDP)  \
Country Year                                     
Belize  1971                         17.381807   
        1972                         17.644155   
        1973                         18.130458   
        1974                         18.418694   
        1975                         18.565922   

              log_Exports of goods and services  \
Country Year                                      
Belize  1971                          16.416311   
        1972                          16.726951   
        1973                          17.016441   
        1974                          17.538038   
        1975                          17.781002   

              log_Imports of goods and services  \
Country Year                                      
Belize  1971                          16.569805   
        1972                          16.871114   
        1973                          17.262787   
        1974                          17.621009   
        1975                          17.901406   

              log_Final consumption expenditure  log_Per capita GNI  \
Country Year                                                          
Belize  1971                          17.085672            5.501258   
        1972                          17.334141            5.752573   
        1973                          17.891319            6.242223   
        1974                          18.076654            6.459904   
        1975                          18.213075            6.590301   

              growth_Gross Domestic Product (GDP)  \
Country Year                                        
Belize  1971                             0.267482   
        1972                             0.299979   
        1973                             0.626292   
        1974                             0.334073   
        1975                             0.158618   

              growth_Gross capital formation  \
Country Year                                   
Belize  1971                        0.248238   
        1972                        0.335795   
        1973                        0.612286   
        1974                        0.249012   
        1975                        0.348679   

              growth_Exports of goods and services  \
Country Year                                         
Belize  1971                              0.306028   
        1972                              0.364298   
        1973                              0.335746   
        1974                              0.684715   
        1975                              0.275023   

              growth_Imports of goods and services  \
Country Year                                         
Belize  1971                              0.273604   
        1972                              0.351626   
        1973                              0.479454   
        1974                              0.430784   
        1975                              0.323655   

              growth_Final consumption expenditure  growth_Per capita GNI  \
Country Year                                                                
Belize  1971                              0.248805               0.256410   
        1972                              0.282061               0.285714   
        1973                              0.745739               0.631746   
        1974                              0.203622               0.243191   
        1975                              0.146164               0.139280   

              lag1_Gross Domestic Product (GDP)  
Country Year                                     
Belize  1971                         27917782.0  
        1972                         35385285.0  
        1973                         46000141.0  
        1974                         74809672.0  
        1975                         99801545.0

## 2.4-Summary - Notebook 2

| Dimension              | Processing Result                                                     | Insight                                                                                      |
| ---------------------- | --------------------------------------------------------------------- | -------------------------------------------------------------------------------------------- |
| Dataset structure      | Balanced country–year panel (7 Central American countries, 1970–2021) | Provides a fully consistent longitudinal structure suitable for dynamic time series analysis |
| Temporal design        | Conversion to PeriodIndex and MultiIndex (Country–Year)               | Ensures strict chronological ordering and correct panel time series behavior                 |
| Missing data structure | Only structural NaNs from lag and growth construction                 | Confirms correct implementation of temporal transformations without data leakage             |
| Feature engineering    | Log transformations applied to positive macro variables               | Reduces skewness and stabilizes variance for macroeconomic comparability                     |
| Growth rates           | Year-over-year percentage changes computed per country                | Captures economic dynamics and cyclical variations across indicators                         |
| Lag features           | 1-year lag variables constructed within each country                  | Introduces temporal memory and enables dynamic dependency modeling                           |
| Data consistency       | No missing values in raw levels; expected NaNs in engineered features | Dataset is structurally valid for econometric and time series modeling                       |
| Feature space design   | Combined logs, growth rates, and lagged GDP in final dataset          | Produces a multi-layered representation balancing interpretability and dynamics              |


## 2.5-Data Export

In [24]:
import os

file_name = "central_america_timeseries_feature_engineering.csv"
df.to_csv(file_name)

print("CSV export completed")
print(f"File saved as: {file_name}")
print("File exists:", os.path.exists(file_name))
print("File size (KB):", round(os.path.getsize(file_name) / 1024, 2))

CSV export completed
File saved as: central_america_timeseries_feature_engineering.csv
File exists: True
File size (KB): 174.8
